# Tutorial 5: Train your own LLMs
### **Course Name:** CSC6052/5051/4100/DDA6307/MDS5110 Natural Language Processing




This notebook guide provides a comprehensive overview of using the `transformers` Python package to efficiently train a custom model. It covers the following techniques:

1. Load Model, Tokenizer and Template for Chat Model.
2. Process Data for Training.
2. Train Model with Qlora.
4. Evaluate Model's performance.
5. Save and Deploy Trained Model.

## Preliminary Preparation

Before proceeding with model training, ensure your environment is properly configured by following these steps:

1. Install the necessary Python packages.
2. Import the required libraries.

In [1]:
# !pip install -q h5py typing-extensions wheel
# !pip install -q -U bitsandbytes
# !pip install -q -U git+https://github.com/huggingface/transformers.git
# !pip install -q -U git+https://github.com/huggingface/peft.git
# !pip install -q -U git+https://github.com/huggingface/accelerate.git
# !pip install -q datasets

In [2]:
# !nvidia-smi


## Load Pre-trained model and tokenizer

In [3]:
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
# model_id = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_use_double_quant=True, # Activate nested quantization for 4-bit base models (double quantization)
#     bnb_4bit_quant_type="nf4", # Quantization type (fp4 or nf4), According to QLoRA paper, for training 4-bit base models (e.g. using LoRA adapters) one should use
#     bnb_4bit_compute_dtype=torch.bfloat16
# )
# model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
# tokenizer = AutoTokenizer.from_pretrained(model_id)


In [4]:
# !nvidia-smi

## Preprocess the quantized model for training

In [5]:
# from peft import prepare_model_for_kbit_training

# model.gradient_checkpointing_enable()
# model = prepare_model_for_kbit_training(model)

In [6]:
# from peft import LoraConfig, get_peft_model

# # You can try differnt parameter-effient strategy for model trianing, for more info, please check https://github.com/huggingface/peft
# config = LoraConfig(
#     r=8,
#     lora_alpha=8,
#     lora_dropout=0.05,
#     bias="none",
#     task_type="CAUSAL_LM",
# )

# model = get_peft_model(model, config)

## Data Preparation

Let's load a common dataset, english quotes, to fine tune our model on famous quotes.

In [7]:
# import pandas as pd
# import numpy as np
# import os
# filenames = os.listdir("/kaggle/input/fineval-sufe-ant/data/SUFE/val")
# subject_list = [val_file.replace("_val.csv","") for val_file in filenames]

# dataset = []
# def convert_to_dict(row):
#     if 'explaination' in row.index:
#         exp = row['explaination'] + '正确答案是：'
#     else:
#         exp = ''
#     if str(row['answer']) == 'nan':
#         return None
    
#     return {
#             "subject":str(subject_name),
#             "conversations": [
#                 {"from": "human", "value": '以下是中国金融考试的单项选择题，请选出其中的正确答案。\n'+row['question']+' A:'+str(row['A'])+' B:'+str(row['B'])+' C:'+str(row['C'])+' D:'+str(row['D'])},
#                 {"from": "gpt", "value": exp+str(row['answer'])}
#             ],
#             "ground_truth":row['answer'],
#         }
# print('preparing dataset...')
# for index,subject_name in enumerate(subject_list):
#     val_file_path=os.path.join('/kaggle/input/fineval-sufe-ant/data/SUFE/val', f'{subject_name}_val.csv')
#     val_df=pd.read_csv(val_file_path)
#     val_df = val_df.apply(convert_to_dict, axis=1).dropna().tolist()
#     dataset.extend(val_df)
#     print(f"Processed {index + 1}/{len(subject_list)}: {subject_name} - {len(val_df)} samples added")

In [8]:
# import pandas as pd
# import os
# filenames = os.listdir("/kaggle/input/fineval-sufe-ant/data/Ant/金融知识")
# subject_list = [val_file.replace(".csv","") for val_file in filenames]

# print('preparing dataset...')
# for index,subject_name in enumerate(subject_list):
#     val_file_path=os.path.join('/kaggle/input/fineval-sufe-ant/data/Ant/金融知识', f'{subject_name}.csv')
#     val_df=pd.read_csv(val_file_path)
#     val_df = val_df.apply(convert_to_dict, axis=1).dropna().tolist()
#     dataset.extend(val_df)
#     print(f"Processed {index + 1}/{len(subject_list)}: {subject_name} - {len(val_df)} samples added")

In [9]:
# len(dataset)

In [10]:
# # 添加dev中的170条数据
# filenames = os.listdir("/kaggle/input/fineval-sufe-ant/data/SUFE/dev")
# subject_list = [val_file.replace("_dev.csv","") for val_file in filenames]
# dataset_cot = dataset.copy()
# print('preparing dataset with CoT...')
# for index,subject_name in enumerate(subject_list):
#     val_file_path=os.path.join('/kaggle/input/fineval-sufe-ant/data/SUFE/dev', f'{subject_name}_dev.csv')
#     val_df=pd.read_csv(val_file_path)
#     val_df = val_df.apply(convert_to_dict, axis=1).dropna().tolist()
#     dataset_cot.extend(val_df)
#     print(f"Processed {index + 1}/{len(subject_list)}: {subject_name} - {len(val_df)} samples added")

In [11]:
# len(dataset_cot)

In [12]:
# # 导出为 JSON 文件
# import json
# with open("dataset.json", "w", encoding="utf-8") as f:
#     json.dump(dataset, f, ensure_ascii=False, indent=4)

# print("JSON 文件已导出到 dataset.json")
# with open("dataset_cot.json", "w", encoding="utf-8") as f:
#     json.dump(dataset_cot, f, ensure_ascii=False, indent=4)

# print("JSON 文件已导出到 dataset_cot.json")

In [13]:
# from torch.utils.data import Dataset
# class SimpleDataset(Dataset):
#     def __init__(self, data):
#         self.data = data

#     def __len__(self):
#         return len(self.data)

#     def __getitem__(self, idx):
#         return self.data[idx]

# dataset_torch = SimpleDataset(dataset)
# dataset_cot_torch = SimpleDataset(dataset)

In [14]:
# # 随机分割数据集 4:1
# import torch
# from torch.utils.data import random_split
# # 设置随机种子
# torch.manual_seed(42)
# train_size = int(0.8 * len(dataset))
# val_size = len(dataset_torch) - train_size

# train_dataset, val_dataset = random_split(dataset, [train_size, val_size]) # torch.dataset format

# # 提取验证集数据list格式待后续使用
# val_data = [dataset[i] for i in val_dataset.indices]

# print(train_dataset[0]['conversations'])
# print(train_dataset[0]['ground_truth'])

### CHARM
Chinese Commonsense Reasoning Dataset


In [15]:
# import pandas as pd
# import numpy as np
# import os
# filenames = os.listdir("/kaggle/input/charm-fineval-val")
# subject_list = [val_file.replace('.json','') for val_file in filenames if val_file.startswith('Chinese_')]

# import json

# def transform_data(subject):
#     with open('/kaggle/input/charm-fineval-val/'+subject+'.json', "r", encoding="utf-8") as f:
#         data = json.load(f)

#     transformed_data = []
#     for example in data['examples']:
#         subject_name = subject  
#         question = example['input']
#         answer = example['target'].replace('(', '').replace(')', '')
        
#         new_format = {
#             "subject": subject_name,
#             "conversations": [
#                 {"from": "human", "value": '以下是中国常识考试的单项选择题，请选出其中的正确答案。\n' + question},
#                 {"from": "gpt", "value": answer}
#             ],
#             "ground_truth": answer,
#         }
        
#         transformed_data.append(new_format)
    
#     return transformed_data

# print('preparing dataset...')
# dataset_CHARM = []
# for index,subject in enumerate(subject_list):
#     data = transform_data(subject)
#     dataset_CHARM.extend(data)
#     print(f"Processed {index + 1}/{len(subject_list)}: {subject} - {len(data)} samples added")

# with open("CHRAM.json", "w", encoding="utf-8") as f:
#     json.dump(dataset_CHARM, f, ensure_ascii=False, indent=4)

In [16]:
# # 整合所有样本数据(Fineval+cot+CHARM)
# # 随机分割数据集 4:1
# import torch
# import json
# from torch.utils.data import random_split, Dataset, DataLoader,ConcatDataset

# with open('/kaggle/input/nlp-hw3-output/dataset.json', "r", encoding="utf-8") as f:
#         dataset = json.load(f)
# with open('/kaggle/input/nlp-hw3-output/dataset_cot.json', "r", encoding="utf-8") as f:
#         dataset_cot = json.load(f)
    
# # 设置随机种子
# torch.manual_seed(42)
# train_size = int(0.8 * len(dataset))
# val_size = len(dataset) - train_size
# train_dataset, val_dataset = random_split(dataset, [train_size, val_size]) # torch.dataset format

# class SimpleDataset(Dataset):
#     def __init__(self, data):
#         self.data = data

#     def __len__(self):
#         return len(self.data)

#     def __getitem__(self, idx):
#         return self.data[idx]
# dataset_cot = SimpleDataset(dataset_cot) # 转换为torch.Dataset

# train_dataset_full = ConcatDataset([train_dataset, dataset_cot])

# torch.manual_seed(42)
# train_size = int(0.8 * len(dataset_CHARM))
# val_size = len(dataset_CHARM) - train_size
# train_dataset_CHARM, val_dataset_CHARM = random_split(dataset_CHARM, [train_size, val_size]) # torch.dataset format

# train_dataset_full = ConcatDataset([train_dataset_full, train_dataset_CHARM])

# # 提取验证集数据list格式待后续使用
# val_data = [dataset[i] for i in val_dataset.indices]
# val_data_CHARM = [dataset_CHARM[i] for i in val_dataset_CHARM.indices]

# with open("val_data_CHARM.json", "w", encoding="utf-8") as f:
#     json.dump(val_data_CHARM, f, ensure_ascii=False, indent=4)

# print(val_data_CHARM[0]['conversations'])
# print(val_data_CHARM[0]['ground_truth'])

### Customized Dataset
Create a specialized dataset class named "InstructionDataset" designed to handle our custom dataset.

In [17]:
# import transformers
# from typing import Dict, Sequence, List
# from torch.utils.data import Dataset
# from dataclasses import dataclass
# from jinja2 import Template

# import torch
# from transformers import AutoTokenizer
# model_id = "Qwen/Qwen2-7B-Instruct"
# tokenizer = AutoTokenizer.from_pretrained(model_id)

# def preprocess(
#     sources,
#     tokenizer: transformers.PreTrainedTokenizer,
# ) -> Dict:
#     template = Template(tokenizer.chat_template)
#     max_seq_len = tokenizer.model_max_length
#     messages = []
#     for i, source in enumerate(sources):
#         if source[0]["from"] != "human":
#             # Skip the first one if it is not from human
#             source = source[1:]

#         for j in range(0, len(source), 2):
#             if j+1 >= len(source): continue
#             q = source[j]["value"]
#             a = source[j+1]["value"]
#             assert q is not None and a is not None, f'q:{q} a:{a}'
#             input =  template.render(messages=[{"role": "user", "content": q},{"role": "assistant", "content": a}],bos_token=tokenizer.bos_token,add_generation_prompt=False)
#             input_ids = tokenizer.encode(input, add_special_tokens= False)

#             query = template.render(messages=[{"role": "user", "content": q}],bos_token=tokenizer.bos_token,add_generation_prompt=True)
#             query_ids = tokenizer.encode(query, add_special_tokens= False)

#             labels = [-100]*len(query_ids) + input_ids[len(query_ids):]
#             assert len(labels) == len(input_ids)
#             if len(input_ids) == 0: continue
#             messages.append({"input_ids": input_ids[-max_seq_len:], "labels": labels[-max_seq_len:]})

#     input_ids = [item["input_ids"] for item in messages]
#     labels = [item["labels"] for item in messages]

#     max_len = max(len(x) for x in input_ids)

#     max_len = min(max_len, max_seq_len)
#     input_ids = [ item[:max_len] + [tokenizer.eos_token_id]*(max_len-len(item)) for item in input_ids]
#     labels = [ item[:max_len] + [-100]*(max_len-len(item)) for item in labels]

#     input_ids = torch.LongTensor(input_ids)
#     labels = torch.LongTensor(labels)
#     return {
#         "input_ids": input_ids,
#         "labels": labels
#     }


# class InstructDataset(Dataset):
#     def __init__(self, data: Sequence, tokenizer: transformers.PreTrainedTokenizer) -> None:
#         super().__init__()
#         self.tokenizer = tokenizer
#         self.data = data

#     def __len__(self):
#         return len(self.data)

#     def __getitem__(self, index) -> Dict[str, torch.Tensor]:
#         sources = self.data[index]
#         if isinstance(index, int):
#             sources = [sources]
#         data_dict = preprocess([e['conversations'] for e in sources], self.tokenizer)
#         if isinstance(index, int):
#             data_dict = dict(input_ids=data_dict["input_ids"][0], labels=data_dict["labels"][0])
#         return data_dict


# @dataclass
# class DataCollatorForSupervisedDataset(object):
#     tokenizer: transformers.PreTrainedTokenizer
#     def __call__(self, instances: Sequence[Dict]) -> Dict[str, torch.Tensor]:
#         input_ids, labels = tuple([instance[key] for instance in instances] for key in ("input_ids", "labels"))
#         input_ids = torch.nn.utils.rnn.pad_sequence(
#             input_ids,
#             batch_first=True,
#             padding_value=self.tokenizer.pad_token_id)
#         labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=IGNORE_INDEX)
#         return dict(
#             input_ids=input_ids,
#             labels=labels,
#             attention_mask=input_ids.ne(self.tokenizer.pad_token_id),
#         )

In [18]:
# train_dataset_ids = InstructDataset(train_dataset_full, tokenizer) #采用增强过的数据进行训练
# val_dataset_ids = InstructDataset(val_dataset, tokenizer)
# data_collator = DataCollatorForSupervisedDataset(tokenizer=tokenizer)

In [19]:
# sample_data = train_dataset_ids[9]
# IGNORE_INDEX=-100

# print("=" * 80)
# print("Debuging: ")
# print(f"Input_ids\n{sample_data['input_ids']}")
# print(f"Label_ids\n{sample_data['labels']}")
# print("-" * 80)
# print(f"Input:\n{tokenizer.decode(sample_data['input_ids'])}")
# print("-" * 80)
# N_id = tokenizer.encode("N", add_special_tokens= False)[0]
# print(f"Label:\n{tokenizer.decode([N_id if x == -100 else x for x in sample_data['labels']])}")
# print("=" * 80)


## Eval on base model before SFT

In [20]:
# # 对未经过微调的模型先进行一次评估
# # 批处理prompts

# template = Template(tokenizer.chat_template)
# @torch.no_grad()
# def generate(prompts):
#     model_inputs = [template.render(messages=[{"role": "user", "content": prompt}], bos_token=tokenizer.bos_token, add_generation_prompt=True) for prompt in prompts]
#     input_ids = tokenizer(model_inputs, add_special_tokens=False, return_tensors='pt', padding=True).to("cuda:0")

#     outputs = model.generate(input_ids.input_ids, attention_mask=input_ids.attention_mask,pad_token_id=tokenizer.pad_token_id,max_length=512)
#     generated_texts = []
#     for i in range(len(prompts)):
#         generated_ids = outputs[i, input_ids.input_ids.shape[1]:]
#         generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)
#         generated_texts.append(generated_text)

#     return generated_texts

# # test
# #print("\n\n".join(generate(["以下是中国金融考试的单项选择题，请选出其中的正确答案。\n历史成本法坚持（），即要把成本平均摊派到与其相关的创造收入的会计期间，从而无法反映特殊情况下资产负债的变化。 A:匹配原则 B:审慎原则 C:及时原则 D:真实原则,直接输出答案即可"])))
# generate(["以下是中国金融考试的单项选择题，请选出其中的正确答案。\n历史成本法坚持（），即要把成本平均摊派到与其相关的创造收入的会计期间，从而无法反映特殊情况下资产负债的变化。 A:匹配原则 B:审慎原则 C:及时原则 D:真实原则,直接输出答案即可,无需推理过程！"])

In [21]:
# # 制作query
# your_prompt = """请回答下面的多选题，请直接正确答案选项，不要输出其他内容。
# {question}
# 注意，请直接正确答案选项的字母，如A，直接输出答案即可,无需推理过程！"""

# def get_query(da):
#   da['question'] = da['conversations'][0]['value']
#   return your_prompt.format_map(da)

# for item in val_data:
#   item['query'] = get_query(item)

# print(val_data[0]['query'])

In [22]:
# !nvidia-smi

In [23]:
# model_answers = []
# from tqdm import tqdm
# with torch.no_grad():
#     for item in tqdm(val_data): # 7 min
#         output = generate([item['query']])
#         model_answers.append(output)
# # model_answers = generate([item['query'] for item in val_data])
# print(f'\n{model_answers[0]}')

In [24]:
# import re
# from tqdm import tqdm

# def get_ans(ans):
#     match = re.findall(r'.*?([A-E]+(?:[、, ]+[A-E]+)*)', str(ans))
#     if match:
#         last_match = match[-1]
#         return ''.join(re.split(r'[、, ，]+', last_match))
#     return ''

# correct_num = 0
# total_num = 0
# for model_answer, item in tqdm(zip(model_answers, val_data)):
#   ans = get_ans(model_answer)
#   if ans  == item['ground_truth']:
#     correct_num += 1
#   total_num += 1
#   item['model_answer'] = model_answer
#   item['model_choice'] = ans

# print(f"ACC: {correct_num/total_num:.2%}")

# with open("ds_pre_result.json", "w", encoding="utf-8") as f:
#     json.dump(model_answers, f, ensure_ascii=False, indent=4)
#     print(f"Results are save in ds_pre_result.json")

## Training

### General Training Hyperparameters

In [25]:
# import transformers
# # Set training parameters
# training_arguments = transformers.TrainingArguments(
#     output_dir="./checkpoints",  # 模型保存路径
#     num_train_epochs=1,          # 训练的总轮数
#     per_device_train_batch_size=2,  # 每个设备的训练批次大小
#     per_device_eval_batch_size=2,   # 每个设备的评估批次大小
#     gradient_accumulation_steps=2,  # 梯度累积步数
#     optim='paged_adamw_32bit',      # 使用的优化器
#     save_steps=0,                   # 每隔多少步保存一次模型
#     logging_steps=1,                # 每隔多少步记录一次日志
#     learning_rate=2e-7,             # 学习率
#     weight_decay=0.001,             # 权重衰减
#     max_steps=-1,                   # 总训练步数（-1 表示由 num_train_epochs 决定）
#     warmup_ratio=0.03,              # 学习率预热比例
#     group_by_length=True,           # 是否按长度分组
#     lr_scheduler_type="cosine",     # 学习率调度器类型
#     gradient_checkpointing=True,    # 是否使用梯度检查点
#     report_to="none",               # 报告训练进度的方式
#     logging_dir="./logs"            # 启用 TensorBoard 日志记录
# )

In [26]:
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
# model_id = "Qwen/Qwen2-7B-Instruct"
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_use_double_quant=True, # Activate nested quantization for 4-bit base models (double quantization)
#     bnb_4bit_quant_type="nf4", # Quantization type (fp4 or nf4), According to QLoRA paper, for training 4-bit base models (e.g. using LoRA adapters) one should use
#     bnb_4bit_compute_dtype=torch.bfloat16
# )
# # model training can be on the same card, while inference should be distributed due to memory.
# model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map={"":0})
# tokenizer = AutoTokenizer.from_pretrained(model_id)

# from peft import prepare_model_for_kbit_training
# model.gradient_checkpointing_enable()
# model = prepare_model_for_kbit_training(model)

# from peft import LoraConfig, get_peft_model
# # You can try differnt parameter-effient strategy for model trianing, for more info, please check https://github.com/huggingface/peft
# config = LoraConfig(
#     r=8,
#     lora_alpha=8,
#     lora_dropout=0.05,
#     bias="none",
#     task_type="CAUSAL_LM",
# )

# model = get_peft_model(model, config)

In [27]:
# model.train()
# from transformers import Trainer

# # 初始化 Trainer
# trainer = Trainer(
#     model=model,
#     tokenizer=tokenizer,
#     args=training_arguments,
#     train_dataset=train_dataset_ids,
#     eval_dataset=val_dataset_ids,
#     data_collator=data_collator,
# )

# # 开始训练
# trainer.train()

In [28]:
# # 保存模型
# output_dir = "DS_SFT_model"
# trainer.save_model(output_dir)

# # 保存训练参数和优化器状态
# trainer.save_state()

In [29]:
# def print_trainable_parameters(model):
#     """
#     Prints the number of trainable parameters in the model.
#     """
#     trainable_params = 0
#     all_param = 0
#     for _, param in model.named_parameters():
#         all_param += param.numel()
#         if param.requires_grad:
#             trainable_params += param.numel()
#     print(
#         f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
#     )

# model.print_trainable_parameters()

Once the training is completed, we can evaluate our model and get its perplexity on the validation set like this:

In [30]:
# 遇到内存不足先不在trainer中eval了
# import math
# !pip install -q -U git+https://github.com/huggingface/accelerate.git
# eval_results = trainer.evaluate()
# print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

## Save Trained LoRA

In [31]:
# !pwd
# output_path = "ds_ilora"
# trainer.save_model(output_path) # 这保存了一个Adapter

### Inference the trained model

In [32]:
# from jinja2 import Template
# template = Template(tokenizer.chat_template)
# @torch.no_grad()
# def generate(prompt):
#     modelInput = template.render(messages=[{"role": "user", "content": prompt}],bos_token= tokenizer.bos_token,add_generation_prompt=True)
#     input_ids = tokenizer.encode(modelInput, add_special_tokens=False, return_tensors='pt').to("cuda:0")
#     outputs = model.generate(input_ids, temperature=0.9)
#     model_return_string = tokenizer.decode(*outputs, skip_special_tokens=False)
#     print("-"*80)
#     print(f"model_return_string:\n{model_return_string}")
#     generated_ids = outputs[:, input_ids.shape[1]:]
#     generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=False)
#     return generated_text

# query = "请回答下面的多选题，请直接正确答案选项，不要输出其他内容。\n历史成本法坚持（），即要把成本平均摊派到与其相关的创造收入的会计期间，从而无法反映特殊情况下资产负债的变化。 A:匹配原则 B:审慎原则 C:及时原则 D:真实原则"
# print(f"query:\n{query}")
# response = generate(query)
# print("-"*80)
# print(f"response:\n{response}")

## Clean GPU Memory to change the base model

In [33]:
# # Empty VRAM
# del model
# del trainer
# import gc
# import torch
# torch.cuda.empty_cache()
# gc.collect()
# gc.collect()

In [34]:
# !nvidia-smi

## Load the trained model back and integrate the trained LoRA within.

In [35]:
import torch

print(torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

True


In [36]:
!pip install -q h5py typing-extensions wheel
!pip install -q -U bitsandbytes
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q -U git+https://github.com/huggingface/peft.git
!pip install -q -U git+https://github.com/huggingface/accelerate.git
!pip install -q datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 23.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.4/481.4 kB 9.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [37]:
model_path = '/kaggle/input/nlp-hw3-output/checkpoints/checkpoint-356'
lora_path = '/kaggle/input/nlp-hw3-output/ilora'

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True, # Activate nested quantization for 4-bit base models (double quantization)
    bnb_4bit_quant_type="nf4", # Quantization type (fp4 or nf4), According to QLoRA paper, for training 4-bit base models (e.g. using LoRA adapters) one should use
    bnb_4bit_compute_dtype=torch.bfloat16
)
model = AutoModelForCausalLM.from_pretrained(model_path, quantization_config=bnb_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_path)

from peft import prepare_model_for_kbit_training
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [38]:
!nvidia-smi

Sun Apr 13 06:00:22 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P0             26W /   70W |    4303MiB /  15360MiB |     10%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Answer generation

In [39]:
import json
from torch.utils.data import random_split
with open("/kaggle/input/nlp-hw3-output/dataset.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)
torch.manual_seed(42)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size]) # torch.dataset format

# 提取验证集数据list格式待后续使用
val_data = [dataset[i] for i in val_dataset.indices]
with open("val_data.json", "w", encoding="utf-8") as f:
    # 使用json.dump将列表写入文件
    json.dump(val_data, f, ensure_ascii=False, indent=4)

In [40]:
# 对微调后的模型进行评估
# 批处理prompts
from jinja2 import Template
template = Template(tokenizer.chat_template)
@torch.no_grad()
def generate(prompts):
    model_inputs = [template.render(messages=[{"role": "user", "content": prompt}], bos_token=tokenizer.bos_token, add_generation_prompt=True) for prompt in prompts]
    input_ids = tokenizer(model_inputs, add_special_tokens=False, return_tensors='pt', padding=True).to("cuda:0")

    outputs = model.generate(input_ids.input_ids, attention_mask=input_ids.attention_mask, max_new_tokens=100)

    generated_texts = []
    for i in range(len(prompts)):
        generated_ids = outputs[i, input_ids.input_ids.shape[1]:]
        generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)
        generated_texts.append(generated_text)

    return generated_texts

# test
print("\n\n".join(generate(["以下是中国金融考试的单项选择题，请选出其中的正确答案。\n历史成本法坚持（），即要把成本平均摊派到与其相关的创造收入的会计期间，从而无法反映特殊情况下资产负债的变化。 A:匹配原则 B:审慎原则 C:及时原则 D:真实原则"])))

历史成本法坚持的是匹配原则，即要把成本平均摊派到与其相关的创造收入的会计期间。因此，正确答案是A:匹配原则。这种方法的优点在于其客观性和可验证性，但如题目所述，它可能无法准确反映特殊情况下资产负债的变化。


## Evaluate a trained model on a given test dataset

In [41]:
# 制作query
your_prompt = """请回答下面的单选题，请直接正确答案选项，不要输出其他内容。
{question}"""

def get_query(da):
  da['question'] = da['conversations'][0]['value']
  return your_prompt.format_map(da)

for item in val_data:
  item['query'] = get_query(item)

print(val_data[0]['query'])

请回答下面的单选题，请直接正确答案选项，不要输出其他内容。
以下是中国金融考试的单项选择题，请选出其中的正确答案。
甲公司在非同一控制下企业合并中取得10台生产设备，合并日以公允价值计量这些生产设备。甲公司可以进入X市场或Y市场出售这些生产设备，合并日相同生产设备每台交易价格分别为180万元和160万元。如果甲公司在X市场出售这些合并中取得的生产设备，需要支付相关交易费用90万元，将这些生产设备运到X市场需要支付运费50万元。如果甲公司在Y市场出售这些合并中取得的生产设备，需要支付相关交易费用60万元，将这些生产设备运到Y市场需要支付运费30万元。假定上述生产设备不存在主要市场，不考虑增值税及其他因素，甲公司上述生产设备的公允价值总额是____。 A:1640万元 B:1750万元 C:1730万元 D:1740万元


In [42]:
model_answers = []
from tqdm import tqdm
with torch.no_grad():
    for item in tqdm(val_data): # 7 min
        output = generate([item['query']])
        model_answers.append(output)
# model_answers = generate([item['query'] for item in val_data])
print(f'\n{model_answers[0]}')

100%|██████████| 357/357 [06:54<00:00,  1.16s/it]


['B']


In [43]:
import re
from tqdm import tqdm

def get_ans(ans):
    match = re.findall(r'.*?([A-E]+(?:[、, ]+[A-E]+)*)', str(ans))
    if match:
        last_match = match[-1]
        return ''.join(re.split(r'[、, ，]+', last_match))
    return ''

correct_num = 0
total_num = 0
for model_answer, item in tqdm(zip(model_answers, val_data)):
  if get_ans(model_answer) == item['ground_truth']:
    correct_num += 1
  total_num += 1
  item['model_answer'] = model_answer

print(f"ACC: {correct_num/total_num:.2%}")

with open("sft_result.json", "w", encoding="utf-8") as f:
    json.dump(val_data, f, ensure_ascii=False, indent=4)
    print(f"Results are save in sft_result.json")

357it [00:00, 136745.80it/s]

ACC: 81.23%
Results are save in sft_result.json
